# CT4110 – Topic 6 Assignment
## Quantum Fourier Transform and Quantum Phase Estimation

In this exercise, you will use Qiskit to construct the Quantum Fourier Transform (QFT), compare a manual circuit with Qiskit's `QFTGate`, verify the inverse transform, and use inverse QFT inside Quantum Phase Estimation (QPE).

For each question, run every checkpoint in order and include the requested circuit diagrams, numerical results, and short explanations. Complete all `YOUR CODE HERE` and `ANSWER: [ ]` sections.

In [ ]:
# ============================================================
# Course Setup – Run This Cell First
# ============================================================

# Install the packages when they are not already available.
try:
    import qiskit
    import qiskit_aer
except ImportError:
    %pip install -q qiskit qiskit-aer matplotlib

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

print("Qiskit setup complete.")

## Question 1 – Construct the Lecture QFT Circuit

For a vector $(x_0,x_1,\ldots,x_{N-1})$, use the lecture convention

$$y_j=\sum_{k=0}^{N-1}x_k e^{-i2\pi jk/N},\qquad j=0,1,\ldots,N-1.$$

For a basis state $|j\rangle$, the QFT used in this exercise is

$$\operatorname{QFT}|j\rangle=\frac{1}{\sqrt{N}}\sum_{k=0}^{N-1}e^{i2\pi jk/N}|k\rangle.$$

The opposite signs are the convention used in the lecture notes. The factor $1/\sqrt{N}$ normalises the quantum state. For three qubits, $N=2^3=8$.

Use the nontrivial input $|001\rangle=|1\rangle$. Follow the lecture labels $j_1=q_0$, $j_2=q_1$, and $j_3=q_2$. Prepare the input by applying $X$ to `q2`.

Build the lecture's three-qubit QFT circuit in this order:

1. apply $H$ to `q0`, then controlled phases $R_2$ from `q1` to `q0` and $R_3$ from `q2` to `q0`;
2. apply $H$ to `q1`, then a controlled phase $R_2$ from `q2` to `q1`;
3. apply $H$ to `q2`;
4. swap `q0` and `q2`.

In [ ]:
# Question 1 — Checkpoint 1: Define the rotations and build the lecture QFT
def theta(k):
    return 2 * np.pi / (2 ** k)

manual_qft = QuantumCircuit(3, name="Manual QFT")
manual_qft.x(2)
manual_qft.barrier(label="prepare |001>")

# Add the QFT gates in the exact order stated above.
# Use theta(2) for R2 and theta(3) for R3.
# Qiskit syntax: manual_qft.h(qubit)
#                manual_qft.cp(angle, control, target)
#                manual_qft.swap(first_qubit, second_qubit)

# YOUR CODE HERE

manual_qft.draw("mpl")

### Convert to Qiskit's displayed bit order

A QFT encodes information in relative phases. Measuring immediately would hide those phases, so use the exact `Statevector(circuit)` method introduced in Topic 4. Before inspecting or measuring the circuit, call `reverse_bits()` to match Qiskit's qubit setting.

In [ ]:
# Question 1 — Checkpoint 2: Match Qiskit's setting and inspect the statevector
qft_qiskit_setting = manual_qft.reverse_bits()
manual_state = Statevector(qft_qiskit_setting)

print("Manual-QFT amplitudes in the Qiskit setting:")
print(np.round(manual_state.data, 3))

# Question 1 — Checkpoint 3: Measure using the Qiskit setting
# Copy qft_qiskit_setting, measure every qubit, then
# transpile and run it on AerSimulator for 1,000 shots. Display its counts
# and histogram. The call to reverse_bits() above must happen before
# measurement is added.
#
# Syntax:
#     measured_qft = qft_qiskit_setting.copy()
#     measured_qft.measure_all()
# YOUR CODE HERE

## Question 2 – Recreate the QFT with Qiskit's `QFTGate`

Recreate the same transformation using Qiskit's `QFTGate` class. Because the circuit from Question 1 was adjusted to Qiskit's setting before inspection, prepare the matching input $|001\rangle$ here by applying $X$ to `q0`, then append `QFTGate(3)` to `[0, 1, 2]`.

Compare the Question 2 statevector with `manual_state` from Question 1 using `Statevector(...).equiv(...)`. This checks equality up to an unobservable global phase and confirms that the manual lecture circuit matches Qiskit's gate implementation.

In [ ]:
# Question 2 — Checkpoint 1: Build the Qiskit QFT circuit
qft_library = QuantumCircuit(3, name="QFT Qiskit")

# Prepare |001> by applying X to q0. Then append QFTGate(3) to
# qubits [0, 1, 2].
# Syntax: qft_library.append(QFTGate(3), [0, 1, 2])
# YOUR CODE HERE

qft_library.decompose().draw("mpl")

In [ ]:
# Question 2 — Checkpoint 2: Compare the two QFT constructions
# Create library_state using the Topic 4 Statevector(circuit) method.
# YOUR CODE HERE

print("Manual circuit matches QFTGate:", manual_state.equiv(library_state))

# What does a result of True demonstrate?
# ANSWER: [ ]

## Question 3 – Reverse the QFT with the Inverse QFT

The inverse QFT is $\operatorname{QFT}^{\dagger}$. The lecture notes use the identity

$$\operatorname{QFT}^{\dagger}\operatorname{QFT}|\psi\rangle=|\psi\rangle.$$

Start from `qft_qiskit_setting`, the manual QFT circuit prepared for Qiskit's setting in Question 1. Append `QFTGate(3).inverse()` to that circuit to reverse the manually constructed QFT. Do **not** start from the `QFTGate` circuit created in Question 2.

First verify the recovery of $|001\rangle$ exactly with a statevector. Then copy the circuit, measure **all three qubits**, and run it on the simulator. Because the round trip is deterministic, the expected result is `001` on every ideal shot; no statistical search for a “most frequent” state is needed.

In [ ]:
# Question 3 — Checkpoint 1: Append inverse QFT to the Question 1 circuit
roundtrip_qc = qft_qiskit_setting.copy()

# The copied circuit already contains the |001> preparation and the manual
# QFT from Question 1. Append only QFTGate(3).inverse() to [0, 1, 2].
# Do not use qft_library from Question 2 and do not add measurements yet.
# YOUR CODE HERE

roundtrip_state = Statevector(roundtrip_qc)
target_state = Statevector.from_label("001")
print("Round trip returns |001>:", roundtrip_state.equiv(target_state))
roundtrip_qc.draw("mpl")

In [ ]:
# Question 3 — Checkpoint 2: Measure every qubit before simulation
measured_roundtrip = roundtrip_qc.copy()

# Add a measurement to every qubit. Then create an AerSimulator, transpile,
# run 1,000 shots, retrieve the counts, and plot the result.
# Syntax: measured_roundtrip.measure_all()
# YOUR CODE HERE

print(counts)
plot_histogram(counts, title="QFT followed by inverse QFT")

# State the expected ideal counts and explain why the result is deterministic.
# ANSWER: [ ]

## Question 4 – Hard Task: Quantum Phase Estimation

Quantum Phase Estimation estimates $\phi$ when an eigenstate satisfies

$$U|u\rangle=e^{i2\pi\phi}|u\rangle.$$

Use three counting qubits and one eigenstate qubit to estimate

$$\phi=\frac18=0.001_2.$$

Prepare the eigenstate qubit in $|1\rangle$. For this eigenstate, a phase gate with angle $2\pi\phi$ supplies the required eigenvalue. Following the endian-aware process above, pair the power $2^j$ with counting qubit `q[n_counting - 1 - j]`. Its controlled-phase angle is

$$2\pi\phi\,2^j.$$

Apply inverse QFT to the counting register in the order `[2, 1, 0]`, measure every counting qubit into the classical bit with the same index, and then use `reverse_bits()` for the final displayed circuit. The expected result represents the integer 1, giving $1/2^3=1/8$.

In [ ]:
# Question 4 — Checkpoint 1: Build and draw the QPE circuit
n_counting = 3
eigenstate_qubit = n_counting
expected_phi = 1 / 8
qpe_qc = QuantumCircuit(n_counting + 1, n_counting)

# 1. Prepare the eigenstate qubit in |1>.
# 2. Apply H to each counting qubit.
# 3. For j = 0, 1, 2, apply the controlled power using:
#    angle = 2*pi*expected_phi*(2**j)
#    control = n_counting - 1 - j
# 4. Append QFTGate(n_counting).inverse() to [2, 1, 0].
# 5. Measure counting qubit j into classical bit j.
# 6. Replace qpe_qc with qpe_qc.reverse_bits() after measurement.
# Add barriers between the preparation, controlled-power, inverse-QFT,
# and measurement stages so the diagram is easy to read.
# YOUR CODE HERE

qpe_qc.draw("mpl")

In [ ]:
# Question 4 — Checkpoint 2: Run QPE and calculate the estimate
shots = 1000

# Transpile qpe_qc for AerSimulator, run it, and retrieve qpe_counts.
# YOUR CODE HERE

plot_histogram(qpe_counts, title="QPE estimate for phi = 1/8")

most_likely = max(qpe_counts, key=qpe_counts.get)
measured_integer = int(most_likely, 2)
estimated_phi = measured_integer / (2 ** n_counting)
absolute_error = abs(estimated_phi - expected_phi)

print("Counts:", qpe_counts)
print("Most likely bit string:", most_likely)
print("Measured integer:", measured_integer)
print("Estimated phase:", estimated_phi)
print("Expected phase:", expected_phi)
print("Absolute error:", absolute_error)

In [ ]:
# Question 4 — Checkpoint 3: Show the phase calculation
# Using the most likely measured bit string, write down:
#
# a) its conversion from binary to an integer x;
# ANSWER: [ ]
#
# b) the calculation phi_estimated = x / 2^n for n = 3;
# ANSWER: [ ]
#
# c) the absolute error |phi_estimated - 1/8|.
# ANSWER: [ ]